# Stage 3 — Clinical NLP + SLM: Exploratory Data Analysis (EDA)
**Project**: Personalized Precision Medicine for Oncology Treatment Optimization  
**Dataset**: `clinical_nlp_dataset_v1.parquet` (6,098 documents, 1,000 unique patients, 2,038 encounters)  
**Execution Mode**: READ-ONLY Analysis (Zero data modification, Zero model training)  
---
### Purpose & Objectives
This notebook provides a complete, reproducible exploratory characterization of the clinical text corpus, evaluating schema cleanliness, token length distributions, clinical vocabulary, negation patterns, patient/encounter clustering, class imbalance, and temporal/outcome leakage guardrails.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to sys.path
SRC_DIR = Path("../src").resolve()
sys.path.insert(0, str(SRC_DIR))

import eda_utils
print("EDA utilities loaded successfully!")

## 1. Load Dataset in Read-Only Mode

In [ ]:
df = eda_utils.load_dataset()
print(f"Loaded dataset shape: {df.shape}")
df.head(3)

## 2. Dataset Overview & Dimensions

In [ ]:
overview = eda_utils.compute_dataset_overview(df)
pd.DataFrame([{
    "Total Documents": overview["total_documents"],
    "Unique Patients": overview["unique_patients"],
    "Unique Encounters": overview["unique_encounters"],
    "Docs / Patient (Mean)": overview["docs_per_patient_mean"],
    "Docs / Encounter (Mean)": overview["docs_per_encounter_mean"],
    "Missing Cells": overview["total_missing_cells"],
    "Exact Duplicate Rows": overview["exact_duplicate_rows"]
}])

## 3. Text Length Distribution

In [ ]:
text_stats = eda_utils.compute_text_length_statistics(df)
pd.DataFrame([text_stats["word_stats"]], index=["Word Statistics"])

## 4. Clinical Vocabulary & Entity Frequencies

In [ ]:
vocab_stats = eda_utils.compute_vocabulary_statistics(df)
print(f"Vocabulary Size: {vocab_stats['vocabulary_size']} unique terms")
print(f"Type-Token Ratio: {vocab_stats['type_token_ratio']}")
print(f"Hapax Legomena: {vocab_stats['hapax_legomena_count']} ({vocab_stats['hapax_percentage']}%)")

clin_terms = eda_utils.compute_clinical_terminology_frequencies(df)
pd.DataFrame(list(clin_terms["antineoplastic_agents"].items()), columns=["Drug", "Count"]).head(10)

## 5. Clinical Negation Analysis

In [ ]:
neg_stats = eda_utils.compute_negation_statistics(df)
print(f"Docs with Negation: {neg_stats['documents_with_negation']} ({neg_stats['percentage_documents_with_negation']}%)")
print(f"Mean Negations per Doc: {neg_stats['mean_negations_per_doc']}")
pd.Series(neg_stats["pattern_prevalence"])

## 6. Target Label Distributions & Imbalance

In [ ]:
lbl_stats = eda_utils.compute_label_distribution(df)
print(f"Urgency Imbalance Ratio: {lbl_stats['urgency_imbalance_ratio']}:1")
pd.DataFrame({
    "Count": lbl_stats["urgency_counts"],
    "Percentage (%)": lbl_stats["urgency_percentages"]
})

## 7. Split Isolation & Patient Overlap Verification

In [ ]:
split_stats = eda_utils.compute_split_statistics(df)
print("Patient Overlap Checks:", split_stats["patient_overlap"])
print("Encounter Overlap Checks:", split_stats["encounter_overlap"])
assert split_stats["patient_overlap"]["is_patient_leakage_free"] is True
assert split_stats["encounter_overlap"]["is_encounter_leakage_free"] is True
print("Split isolation verified with ZERO leakage!")

## 8. Leakage Risk Audit

In [ ]:
leakage = eda_utils.analyze_leakage_risks(df)
print("Leakage Risk Classification:", leakage["leakage_risk_classification"])
print("Forbidden terms detected:", leakage["detections_per_phrase"])